# Bronze Orchestrator: master_products
Orchestrates ingestion, validation, and monitoring for the Bronze layer of master_products.

**Execution Order:**
1. Ingestion
2. Validation
3. Monitoring

**Alerts:**
- Execution errors are captured and displayed for each step.
- If any step fails, subsequent steps are not executed.

In [ ]:
import sys
sys.path.append("/Workspace/Users/diego.mayorgacapera@gmail.com/.bundle/BI_Market_Visibility/dev/files")
# Import orchestrated functions
from src.bronze.master_products.ingest_master_products import run_ingestion
from src.bronze.master_products.validate_master_products import run_validation
from src.bronze.master_products.monitor_master_products import run_monitoring

In [ ]:
# --- Environment Parameters ---
# You can change these for staging/prod
# Bronze table and validation table names must match your deployment
# Example: env = 'prod'
# Example: bronze_table = 'workspace.bronze.master_products'
# Example: validation_table = 'workspace.bronze.master_products_validation'
env = 'dev'
bronze_table = 'workspace.bronze.master_products'
validation_table = 'workspace.bronze.master_products_validation'

In [ ]:
# Step 1: Ingestion
try:
    batch_id, rows_ingested = run_ingestion(source_path='/Volumes/workspace/raw_data/master_products', delta_table=bronze_table, env=env)
    print(f'✅ Ingestion completed. Batch ID: {batch_id}, Rows: {rows_ingested}')
except Exception as e:
    print(f'❌ Ingestion failed: {str(e)}')
    raise

In [ ]:
# Step 2: Validation (only if ingestion succeeded)
try:
    metrics_df = run_validation(bronze_table=bronze_table, env=env)
    print('✅ Validation completed.')
    display(metrics_df)
except Exception as e:
    print(f'❌ Validation failed: {str(e)}')
    raise

In [ ]:
# Step 3: Monitoring (only if validation succeeded)
try:
    metrics_json, alerts_json = run_monitoring(bronze_table=bronze_table, validation_table=validation_table, env=env)
    print('✅ Monitoring completed.')
    print('Metrics:')
    print(metrics_json)
    print(f'Metrics volume: {len(metrics_json)}')
    if alerts_json:
        print('⚠️ Alerts:')
        print(alerts_json)
        print(f'Alerts volume: {len(alerts_json)}')
    else:
        print('No alerts triggered.')
except Exception as e:
    print(f'❌ Monitoring failed: {str(e)}')
    raise